# Baseline vs. L2 + Dropout: tuning and test evaluation

A compact re-implementation of the same task with small reusable functions. It tunes L2 (λ) and Dropout rate on a validation split and then evaluates a baseline model and the best regularized model on the **test set**.

The notebook contains two early script-style drafts followed by the final modular version. They use different settings, so their numbers are not comparable with each other; the **final version** (from the `Final version` heading downwards) uses the same architecture, optimizer and batch size as notebook 01.


## Draft 1 (script version)

Baseline 64-64, RMSprop; validation split = first 1,000 training samples; tuning for 10 epochs per setting.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from keras.datasets import reuters
from keras import models, layers
from keras.utils import to_categorical
from keras.regularizers import l2

# Theoretical Note:
# Overfitting occurs when a model learns noise in the training data, leading to poor generalization.
# Regularization techniques like L2 weight regularization and Dropout mitigate overfitting by:
# - L2 regularization: Adds a penalty on large weights, encouraging simpler models.
# - Dropout: Randomly drops neurons during training, promoting robust feature learning.
# Hyperparameter tuning (e.g., L2 lambda, dropout rate) on validation data ensures optimal regularization.

# Step 1: Load and Preprocess Data (Using Provided Code)
# Load Reuters dataset
(train_data, train_labels), (test_data, test_labels) = reuters.load_data(num_words=10000)

# Vectorize the sequences
def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1
    return results

x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)

# Convert labels to one-hot encoding
one_hot_train_labels = to_categorical(train_labels)
one_hot_test_labels = to_categorical(test_labels)

# Split the data into training and validation sets
x_val = x_train[:1000]
partial_x_train = x_train[1000:]
y_val = one_hot_train_labels[:1000]
partial_y_train = one_hot_train_labels[1000:]

# Step 2: Define Models
def build_base_model():
    """
    Build a model without regularization (prone to overfitting).
    Matches the architecture of the provided base code.
    """
    model = models.Sequential()
    model.add(layers.Dense(64, activation='relu', input_shape=(10000,)))
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dense(46, activation='softmax'))
    model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

def build_regularized_model(l2_lambda=0.01, dropout_rate=0.5):
    """
    Build a model with L2 regularization and Dropout.
    l2_lambda: L2 regularization strength.
    dropout_rate: Fraction of units to drop.
    """
    model = models.Sequential()
    model.add(layers.Dense(64, activation='relu', input_shape=(10000,), kernel_regularizer=l2(l2_lambda)))
    model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(64, activation='relu', kernel_regularizer=l2(l2_lambda)))
    model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(46, activation='softmax'))
    model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Step 3: Train and Evaluate Model
def train_evaluate_model(model, x_train, y_train, x_val, y_val, epochs=20, batch_size=512):
    """
    Train the model and return history for analysis.
    """
    history = model.fit(
        x_train, y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(x_val, y_val),
        verbose=1
    )
    return history

# Step 4: Hyperparameter Tuning for Regularization
def tune_regularization(x_train, y_train, x_val, y_val):
    """
    Tune L2 lambda and dropout rate using validation data.
    Returns the best model and hyperparameters based on validation accuracy.
    """
    l2_lambdas = [0.001, 0.01, 0.1]
    dropout_rates = [0.3, 0.5]
    best_val_acc = 0
    best_lambda = None
    best_dropout = None
    best_model = None
    best_history = None
    
    for l2_lambda in l2_lambdas:
        for dropout_rate in dropout_rates:
            print(f"\nTesting L2_lambda={l2_lambda}, Dropout_rate={dropout_rate}")
            model = build_regularized_model(l2_lambda, dropout_rate)
            history = train_evaluate_model(model, x_train, y_train, x_val, y_val, epochs=10)
            
            # Get the best validation accuracy
            val_acc = max(history.history['val_accuracy'])
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_lambda = l2_lambda
                best_dropout = dropout_rate
                best_model = model
                best_history = history
                
    print(f"\nBest hyperparameters: L2_lambda={best_lambda}, Dropout_rate={best_dropout}")
    return best_model, best_history, best_lambda, best_dropout

# Step 5: Plot Training and Validation Metrics
def plot_metrics(base_history, reg_history, best_lambda, best_dropout):
    """
    Plot training/validation loss and accuracy to compare base and regularized models.
    """
    plt.figure(figsize=(12, 5))
    
    # Plot Loss
    plt.subplot(1, 2, 1)
    plt.plot(base_history.history['loss'], label='Base Train Loss')
    plt.plot(base_history.history['val_loss'], label='Base Val Loss')
    plt.plot(reg_history.history['loss'], label='Reg Train Loss')
    plt.plot(reg_history.history['val_loss'], label='Reg Val Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    # Plot Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(base_history.history['accuracy'], label='Base Train Accuracy')
    plt.plot(base_history.history['val_accuracy'], label='Base Val Accuracy')
    plt.plot(reg_history.history['accuracy'], label='Reg Train Accuracy')
    plt.plot(reg_history.history['val_accuracy'], label='Reg Val Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    
    plt.suptitle(f'Regularized Model (L2_lambda={best_lambda}, Dropout={best_dropout})')
    plt.tight_layout()
    plt.savefig('model_comparison.png')
    plt.show()

# Step 6: Main Execution
def main():
    # Train base model (no regularization)
    print("\nTraining Base Model (No Regularization)")
    base_model = build_base_model()
    base_history = train_evaluate_model(
        base_model, partial_x_train, partial_y_train, x_val, y_val
    )
    
    # Tune and train regularized model
    print("\nTuning Regularized Model")
    reg_model, reg_history, best_lambda, best_dropout = tune_regularization(
        partial_x_train, partial_y_train, x_val, y_val
    )
    
    # Evaluate both models on test data
    base_results = base_model.evaluate(x_test, one_hot_test_labels, verbose=0)
    reg_results = reg_model.evaluate(x_test, one_hot_test_labels, verbose=0)
    
    print(f"\nBase Model Test Loss: {base_results[0]:.4f}, Test Accuracy: {base_results[1]:.4f}")
    print(f"Regularized Model Test Loss: {reg_results[0]:.4f}, Test Accuracy: {reg_results[1]:.4f}")
    
    # Plot comparison
    plot_metrics(base_history, reg_history, best_lambda, best_dropout)
    
    # Analysis:
    # The base model typically overfits, with high training accuracy but lower validation/test accuracy.
    # The regularized model, with tuned L2 and Dropout, shows closer train/validation curves,
    # indicating better generalization. The plots visually highlight the overfitting in the base model
    # and the improved performance of the regularized model.

if __name__ == "__main__":
    main()


Training Base Model (No Regularization)


Epoch 1/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.3578 - loss: 3.2290 - val_accuracy: 0.6190 - val_loss: 1.8235
Epoch 2/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6653 - loss: 1.6189 - val_accuracy: 0.7060 - val_loss: 1.3459
Epoch 3/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7365 - loss: 1.1984 - val_accuracy: 0.7490 - val_loss: 1.1440
Epoch 4/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7926 - loss: 0.9469 - val_accuracy: 0.7610 - val_loss: 1.0447
Epoch 5/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8355 - loss: 0.7744 - val_accuracy: 0.7940 - val_loss: 0.9670
Epoch 6/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8675 - loss: 0.6506 - val_accuracy: 0.8070 - val_loss: 0.9263
Epoch 7/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8935 - loss: 0.5222 - val_accuracy: 0.8200 - val_loss: 0.8854
Epoch 8/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9115 - loss: 0.4309 - val_accuracy: 0.8010 - v

## Draft 2 (script version)

Different architecture (512-256, Adam, batch size 32) and a random 20% validation split. Not comparable with Draft 1 or the final version.


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import reuters
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# Theoretical Note:
# Overfitting occurs when a model learns patterns specific to the training data, including noise,
# leading to poor generalization on unseen data. Regularization techniques like L2 weight
# regularization and Dropout help prevent overfitting by:
# - L2 regularization: Adds a penalty on the size of weights, encouraging simpler models.
# - Dropout: Randomly drops neurons during training, forcing the network to learn robust features.
# Hyperparameter tuning (e.g., L2 lambda, dropout rate) is critical to balance underfitting and overfitting.

# Step 1: Load and Preprocess the Reuters Dataset
def load_preprocess_data(num_words=10000, maxlen=200):
    """
    Load and preprocess Reuters dataset.
    num_words: Max number of words to consider (vocabulary size).
    maxlen: Max length of each sequence.
    """
    # Load data
    (x_train, y_train), (x_test, y_test) = reuters.load_data(num_words=num_words)
    
    # Vectorize sequences (truncate/pad to maxlen)
    def vectorize_sequences(sequences, dimension=num_words):
        results = np.zeros((len(sequences), dimension))
        for i, sequence in enumerate(sequences):
            results[i, sequence] = 1.0  # One-hot encoding
        return results
    
    x_train = vectorize_sequences(x_train)
    x_test = vectorize_sequences(x_test)
    
    # Convert labels to one-hot encoding
    num_classes = np.max(y_train) + 1
    y_train = to_categorical(y_train, num_classes)
    y_test = to_categorical(y_test, num_classes)
    
    # Split training data into train and validation
    x_train, x_val, y_train, y_val = train_test_split(
        x_train, y_train, test_size=0.2, random_state=42
    )
    
    return x_train, y_train, x_val, y_val, x_test, y_test, num_classes

# Step 2: Build Models
def build_base_model(input_dim, num_classes):
    """
    Build a model without regularization (prone to overfitting).
    """
    model = Sequential([
        Dense(512, activation='relu', input_shape=(input_dim,)),
        Dense(256, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

def build_regularized_model(input_dim, num_classes, l2_lambda=0.01, dropout_rate=0.5):
    """
    Build a model with L2 regularization and Dropout.
    l2_lambda: L2 regularization strength.
    dropout_rate: Fraction of units to drop.
    """
    model = Sequential([
        Dense(512, activation='relu', input_shape=(input_dim,), kernel_regularizer=l2(l2_lambda)),
        Dropout(dropout_rate),
        Dense(256, activation='relu', kernel_regularizer=l2(l2_lambda)),
        Dropout(dropout_rate),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Step 3: Train and Evaluate Model
def train_evaluate_model(model, x_train, y_train, x_val, y_val, epochs=20, batch_size=32):
    """
    Train the model and return history for analysis.
    """
    history = model.fit(
        x_train, y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(x_val, y_val),
        verbose=1
    )
    return history

# Step 4: Hyperparameter Tuning for Regularization
def tune_regularization(x_train, y_train, x_val, y_val, input_dim, num_classes):
    """
    Tune L2 lambda and dropout rate using validation data.
    Returns the best hyperparameters based on validation accuracy.
    """
    l2_lambdas = [0.001, 0.01, 0.1]
    dropout_rates = [0.3, 0.5]
    best_val_acc = 0
    best_lambda = None
    best_dropout = None
    best_model = None
    best_history = None
    
    for l2_lambda in l2_lambdas:
        for dropout_rate in dropout_rates:
            print(f"\nTesting L2_lambda={l2_lambda}, Dropout_rate={dropout_rate}")
            model = build_regularized_model(input_dim, num_classes, l2_lambda, dropout_rate)
            history = train_evaluate_model(model, x_train, y_train, x_val, y_val, epochs=10)
            
            # Get the best validation accuracy
            val_acc = max(history.history['val_accuracy'])
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_lambda = l2_lambda
                best_dropout = dropout_rate
                best_model = model
                best_history = history
                
    print(f"\nBest hyperparameters: L2_lambda={best_lambda}, Dropout_rate={best_dropout}")
    return best_model, best_history, best_lambda, best_dropout

# Step 5: Plot Training and Validation Metrics
def plot_metrics(base_history, reg_history, best_lambda, best_dropout):
    """
    Plot training/validation loss and accuracy for both models.
    """
    plt.figure(figsize=(12, 5))
    
    # Plot Loss
    plt.subplot(1, 2, 1)
    plt.plot(base_history.history['loss'], label='Base Train Loss')
    plt.plot(base_history.history['val_loss'], label='Base Val Loss')
    plt.plot(reg_history.history['loss'], label='Reg Train Loss')
    plt.plot(reg_history.history['val_loss'], label='Reg Val Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    # Plot Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(base_history.history['accuracy'], label='Base Train Accuracy')
    plt.plot(base_history.history['val_accuracy'], label='Base Val Accuracy')
    plt.plot(reg_history.history['accuracy'], label='Reg Train Accuracy')
    plt.plot(reg_history.history['val_accuracy'], label='Reg Val Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    
    plt.suptitle(f'Regularized Model (L2_lambda={best_lambda}, Dropout={best_dropout})')
    plt.tight_layout()
    plt.savefig('model_comparison.png')
    plt.close()

# Step 6: Main Execution
def main():
    # Load and preprocess data
    num_words = 10000
    x_train, y_train, x_val, y_val, x_test, y_test, num_classes = load_preprocess_data(num_words)
    
    # Train base model (no regularization)
    print("\nTraining Base Model (No Regularization)")
    base_model = build_base_model(num_words, num_classes)
    base_history = train_evaluate_model(base_model, x_train, y_train, x_val, y_val)
    
    # Tune and train regularized model
    print("\nTuning Regularized Model")
    reg_model, reg_history, best_lambda, best_dropout = tune_regularization(
        x_train, y_train, x_val, y_val, num_words, num_classes
    )
    
    # Evaluate both models on test data
    base_test_loss, base_test_acc = base_model.evaluate(x_test, y_test, verbose=0)
    reg_test_loss, reg_test_acc = reg_model.evaluate(x_test, y_test, verbose=0)
    
    print(f"\nBase Model Test Accuracy: {base_test_acc:.4f}")
    print(f"Regularized Model Test Accuracy: {reg_test_acc:.4f}")
    
    # Plot comparison
    plot_metrics(base_history, reg_history, best_lambda, best_dropout)
    
    # Analysis:
    # The base model is expected to overfit, showing high training accuracy but lower validation/test accuracy.
    # The regularized model, with tuned L2 and Dropout, should show closer train/validation curves,
    # indicating better generalization. The plots will visually confirm this.

if __name__ == "__main__":
    main()


Training Base Model (No Regularization)
Epoch 1/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - accuracy: 0.6362 - loss: 1.7428 - val_accuracy: 0.8180 - val_loss: 0.8594
Epoch 2/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 0.9145 - loss: 0.3684 - val_accuracy: 0.8286 - val_loss: 0.8022
Epoch 3/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - accuracy: 0.9552 - loss: 0.1844 - val_accuracy: 0.8069 - val_loss: 0.9006
Epoch 4/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 0.9644 - loss: 0.1297 - val_accuracy: 0.8264 - val_loss: 0.8808
Epoch 5/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 0.9614 - loss: 0.1240 - val_accuracy: 0.8225 - val_loss: 0.9280
Epoch 6/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 0.9616 - loss: 0.1071 - val_accuracy: 0.8136 - val_loss: 1.0239
Epoch 7/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 0.9664 - loss: 0.0842 - val_accuracy: 0.8264 - val_loss: 0.9681
Epoch 8/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - a

## Final version

Baseline 64-64, RMSprop, batch size 512, validation split with `train_test_split(test_size=1000, random_state=42)`; grid over L2 ∈ {0.001, 0.01, 0.1} × Dropout ∈ {0.3, 0.5}, 20 epochs per setting.


In [4]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import reuters
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
def load_preprocess_data(num_words=10000):
    (x_train, y_train), (x_test, y_test) = reuters.load_data(num_words=num_words)

    def vectorize_sequences(sequences, dimension=num_words):
        results = np.zeros((len(sequences), dimension))
        for i, sequence in enumerate(sequences):
            results[i, sequence] = 1.0
        return results

    x_train = vectorize_sequences(x_train)
    x_test = vectorize_sequences(x_test)

    y_train = to_categorical(y_train)
    y_test = to_categorical(y_test)

    x_train, x_val, y_train, y_val = train_test_split(
        x_train, y_train, test_size=1000, random_state=42
    )

    return x_train, y_train, x_val, y_val, x_test, y_test

x_train, y_train, x_val, y_val, x_test, y_test = load_preprocess_data()
def build_base_model(input_dim=10000, num_classes=46):
    model = Sequential([
        Dense(64, activation='relu', input_shape=(input_dim,)),
        Dense(64, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

base_model = build_base_model()


In [5]:
def build_regularized_model(input_dim, num_classes, l2_lambda=0.01, dropout_rate=0.5):
    model = Sequential([
        Dense(64, activation='relu', input_shape=(input_dim,), kernel_regularizer=l2(l2_lambda)),
        Dropout(dropout_rate),
        Dense(64, activation='relu', kernel_regularizer=l2(l2_lambda)),
        Dropout(dropout_rate),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


In [6]:
def train_model(model, x_train, y_train, x_val, y_val, epochs=20, batch_size=512):
    history = model.fit(
        x_train, y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(x_val, y_val),
        verbose=1
    )
    return history
print("🔹 Training Base Model")
base_history = train_model(base_model, x_train, y_train, x_val, y_val)


🔹 Training Base Model
Epoch 1/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.3437 - loss: 3.2810 - val_accuracy: 0.6100 - val_loss: 1.9125
Epoch 2/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6492 - loss: 1.6943 - val_accuracy: 0.6920 - val_loss: 1.4319
Epoch 3/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7291 - loss: 1.2654 - val_accuracy: 0.7370 - val_loss: 1.2255
Epoch 4/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7863 - loss: 0.9972 - val_accuracy: 0.7610 - val_loss: 1.1103
Epoch 5/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8250 - loss: 0.8204 - val_accuracy: 0.7810 - val_loss: 1.0220
Epoch 6/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8552 - loss: 0.6861 - val_accuracy: 0.7810 - val_loss: 1.0329
Epoch 7/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8738 - loss: 0.5808 - val_accuracy: 0.7920 - val_loss: 0.9259
Epoch 8/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9051 - loss: 0.4674 - va

In [7]:
def tune_regularization(x_train, y_train, x_val, y_val, input_dim, num_classes):
    l2_lambdas = [0.001, 0.01, 0.1]
    dropout_rates = [0.3, 0.5]
    best_val_acc = 0
    best_model = None
    best_history = None
    best_lambda = None
    best_dropout = None

    for l2_lambda in l2_lambdas:
        for dropout_rate in dropout_rates:
            print(f"\nTrying L2={l2_lambda}, Dropout={dropout_rate}")
            model = build_regularized_model(input_dim, num_classes, l2_lambda, dropout_rate)
            history = train_model(model, x_train, y_train, x_val, y_val, epochs=20)
            val_acc = max(history.history['val_accuracy'])
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_model = model
                best_history = history
                best_lambda = l2_lambda
                best_dropout = dropout_rate

    print(f"\n✅ Best Hyperparameters: L2={best_lambda}, Dropout={best_dropout}")
    return best_model, best_history, best_lambda, best_dropout

reg_model, reg_history, best_lambda, best_dropout = tune_regularization(
    x_train, y_train, x_val, y_val, input_dim=10000, num_classes=46
)



Trying L2=0.001, Dropout=0.3
Epoch 1/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.2998 - loss: 3.5170 - val_accuracy: 0.5920 - val_loss: 2.1039
Epoch 2/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5744 - loss: 2.1267 - val_accuracy: 0.6740 - val_loss: 1.6636
Epoch 3/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6499 - loss: 1.7012 - val_accuracy: 0.7120 - val_loss: 1.4812
Epoch 4/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6989 - loss: 1.4783 - val_accuracy: 0.7390 - val_loss: 1.3705
Epoch 5/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7382 - loss: 1.3247 - val_accuracy: 0.7540 - val_loss: 1.3045
Epoch 6/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7594 - loss: 1.2292 - val_accuracy: 0.7570 - val_loss: 1.2611
Epoch 7/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7698 - loss: 1.1692 - val_accuracy: 0.7720 - val_loss: 1.2121
Epoch 8/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7928 - loss: 1.0

In [13]:
all_histories = {}
l2_lambdas = [0.001, 0.01, 0.1]
dropout_rates = [0.3, 0.5]

best_val_acc = 0
best_model = None
best_lambda = None
best_dropout = None

for l2_lambda in l2_lambdas:
    for dropout_rate in dropout_rates:
        print(f"\nTraining model with L2={l2_lambda}, Dropout={dropout_rate}")
        model = build_regularized_model(10000, 46, l2_lambda, dropout_rate)
        history = train_model(model, x_train, y_train, x_val, y_val, epochs=20)
        key = f"L2={l2_lambda}, Dropout={dropout_rate}"
        all_histories[key] = history

        val_acc = max(history.history['val_accuracy'])
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model = model
            best_lambda = l2_lambda
            best_dropout = dropout_rate
            best_history = history



Training model with L2=0.001, Dropout=0.3
Epoch 1/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.3191 - loss: 3.5495 - val_accuracy: 0.5930 - val_loss: 2.1478
Epoch 2/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5627 - loss: 2.1517 - val_accuracy: 0.6420 - val_loss: 1.7021
Epoch 3/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6369 - loss: 1.7180 - val_accuracy: 0.7010 - val_loss: 1.5134
Epoch 4/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6904 - loss: 1.5041 - val_accuracy: 0.7220 - val_loss: 1.4170
Epoch 5/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7308 - loss: 1.3422 - val_accuracy: 0.7310 - val_loss: 1.3326
Epoch 6/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7505 - loss: 1.2917 - val_accuracy: 0.7480 - val_loss: 1.2699
Epoch 7/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7803 - loss: 1.1443 - val_accuracy: 0.7620 - val_loss: 1.2311
Epoch 8/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.785

In [14]:
def evaluate_model(model, x_test, y_test):
    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    print(f"Test Loss: {loss:.4f}, Test Accuracy: {acc:.4f}")
    return loss, acc

print("\n🔹 Evaluation on Test Data")
evaluate_model(base_model, x_test, y_test)
evaluate_model(reg_model, x_test, y_test)



🔹 Evaluation on Test Data
Test Loss: 0.9973, Test Accuracy: 0.7979
Test Loss: 1.1683, Test Accuracy: 0.7930


(1.1683205366134644, 0.792965292930603)

In [15]:
def plot_all_histories(base_history, all_histories):
    plt.figure(figsize=(14, 6))

    # نمودار Loss
    plt.subplot(1, 2, 1)
    plt.plot(base_history.history['val_loss'], label='Base Model')
    for label, hist in all_histories.items():
        plt.plot(hist.history['val_loss'], label=label)
    plt.title('Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend(loc='best')

    # نمودار Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(base_history.history['val_accuracy'], label='Base Model')
    for label, hist in all_histories.items():
        plt.plot(hist.history['val_accuracy'], label=label)
    plt.title('Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend(loc='best')

    plt.suptitle("Comparison of Base Model and Regularized Models")
    plt.tight_layout()
    plt.show()

plot_all_histories(base_history, all_histories)
